# 🔬 Notebook 3: LinkedIn Connections — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/linkedin-connections
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### BFS for degrees of separation

Given two users u and v, how many connection-hops between them? BFS from u until we hit v (or exceed a depth limit like 3, since the world is small).

In [ ]:
from collections import deque, defaultdict

adj = defaultdict(set)
for a,b in [(1,2),(2,3),(3,4),(1,5),(5,3),(4,6)]:
    adj[a].add(b); adj[b].add(a)

def degree(u, v, max_depth=3):
    if u == v: return 0
    seen, frontier = {u}, deque([(u,0)])
    while frontier:
        node, d = frontier.popleft()
        if d >= max_depth: continue
        for nb in adj[node]:
            if nb == v: return d+1
            if nb not in seen:
                seen.add(nb); frontier.append((nb, d+1))
    return None    # > max_depth

print("1→4:", degree(1,4))
print("1→6:", degree(1,6))
print("1→99:", degree(1,99))

## Deep dive 2

### People You May Know

Classic heuristic: users with many **common 1st-degree** friends are likely to know each other (2nd-degree friends with high overlap). We score candidates by `|N(u) ∩ N(c)|`.

In [ ]:
def pymk(u, adj, k=5):
    first = adj[u]
    scores = {}
    for f in first:
        for fof in adj[f]:
            if fof == u or fof in first: continue
            scores[fof] = scores.get(fof, 0) + 1
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

print("pymk(1):", pymk(1, adj))

## Closing thoughts

- The **graph** is the product — design your storage to match the top queries.
- Cap BFS depth; beyond 3–4 the network is basically 'everyone'.
- **Precompute PYMK** offline; serve from a KV cache.